# ML Enhancement: Market Basket Analysis (FP-Growth)

Discover products frequently purchased together using the FP-Growth algorithm.

## Business Value
- Cross-sell recommendations ("Customers also bought")
- Optimize store layouts based on co-purchase patterns
- Bundle pricing opportunities
- Promotion targeting with complementary products

## Data Flow
```
Configured Silver receipt tables --> FP-Growth --> Configured Gold association outputs
```

## Usage
Schedule this notebook on the cadence that fits your refresh requirements.

## Output
- Association rules with support, confidence, and lift are saved to the configured gold table
- Defaults are configurable via environment variables (support, confidence, receipt type filter, and top-N limit)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timezone
import os
import mlflow


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="market_basket")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
PRODUCT_ASSOCIATIONS_TABLE = get_env("PRODUCT_ASSOCIATIONS_TABLE", default="product_associations")
PRODUCT_RECOMMENDATIONS_TABLE = get_env("PRODUCT_RECOMMENDATIONS_TABLE", default="product_recommendations")
PRODUCT_ASSOCIATIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRODUCT_ASSOCIATIONS_TABLE}"
PRODUCT_RECOMMENDATIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRODUCT_RECOMMENDATIONS_TABLE}"
RECEIPT_TYPE_FILTER = get_env("RECEIPT_TYPE_FILTER", default="SALE")

# FP-Growth parameters from acceptance criteria
MIN_SUPPORT = float(get_env("MIN_SUPPORT", default="0.01"))  # 1% of transactions
MIN_CONFIDENCE = float(get_env("MIN_CONFIDENCE", default="0.3"))  # 30%
TOP_N_RULES = int(get_env("TOP_N_RULES", default="100"))  # Top 100 by lift



print(f"Configuration:")
print(f"  SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"  Source tables: {RECEIPTS_TABLE}, {RECEIPT_LINES_TABLE}")
print(f"  Receipt type filter: {RECEIPT_TYPE_FILTER}")
print(f"  Output tables: {PRODUCT_ASSOCIATIONS_TABLE_NAME}, {PRODUCT_RECOMMENDATIONS_TABLE_NAME}")
print(f"  MIN_SUPPORT={MIN_SUPPORT}, MIN_CONFIDENCE={MIN_CONFIDENCE}")
print(f"  TOP_N_RULES={TOP_N_RULES}")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('fact_receipts', 'fact_receipt_lines')
ML_OUTPUT_CONTRACTS = {'product_associations': [('antecedent', 'array<long>', False),
                          ('consequent', 'array<long>', False),
                          ('support', 'double', False),
                          ('confidence', 'double', False),
                          ('lift', 'double', False),
                          ('computed_at', 'timestamp', False)],
 'product_recommendations': [('product_id', 'long', False),
                             ('recommended_product_id', 'long', False),
                             ('support', 'double', False),
                             ('confidence', 'double', False),
                             ('lift', 'double', False),
                             ('computed_at', 'timestamp', False)]}


In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def save_gold(df, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    row_count = spark.table(full_name).count()
    print(f"  {full_name}: {row_count} rows")
    return full_name, row_count

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def canonicalize_itemset(column):
    return F.sort_array(F.array_distinct(column))

ensure_database(GOLD_DB)

mlflow.set_experiment(EXPERIMENT_NAME)

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



In [ ]:
print("="*60)
print("MARKET BASKET ANALYSIS - TRANSACTION PREPARATION")
print("="*60)

In [ ]:
# Step 1: Prepare transaction baskets
# Join fact_receipts with fact_receipt_lines to get all items per receipt

if not silver_exists(RECEIPTS_TABLE) or not silver_exists(RECEIPT_LINES_TABLE):
    raise RuntimeError(
        f"Required tables {RECEIPTS_TABLE} and {RECEIPT_LINES_TABLE} not found in Silver"
    )

print("\nPreparing transaction baskets...")

receipts = (
    read_silver(RECEIPTS_TABLE)
    .filter(F.col("receipt_type") == RECEIPT_TYPE_FILTER)
    .select("receipt_id_ext", "store_id", "event_ts")
)

receipt_lines = (
    read_silver(RECEIPT_LINES_TABLE)
    .select("receipt_id_ext", "product_id")
)

baskets = (
    receipts
    .join(receipt_lines, "receipt_id_ext")
    .groupBy("receipt_id_ext")
    .agg(
        canonicalize_itemset(F.collect_set("product_id")).alias("items")
    )
    .filter(F.size(F.col("items")) > 1)
)

total_baskets = baskets.count()
print(f"  Total transaction baskets (multi-item): {total_baskets:,}")

if total_baskets == 0:
    print("  No multi-item transactions found; empty outputs will replace prior results.")

In [ ]:
print("\n" + "="*60)
print("FP-GROWTH MODEL TRAINING")
print("="*60)

model = None
if total_baskets > 0:
    fpGrowth = FPGrowth(
        itemsCol="items",
        minSupport=MIN_SUPPORT,
        minConfidence=MIN_CONFIDENCE
    )

    print(f"\nTraining FP-Growth model with {total_baskets:,} baskets...")
    model = fpGrowth.fit(baskets)
    print("  Model training complete")
else:
    print("\nSkipping model training because there are no eligible baskets.")

In [ ]:
print("\n" + "="*60)
print("ASSOCIATION RULE EXTRACTION")
print("="*60)

from pyspark.sql.types import ArrayType, DoubleType, LongType, StructField, StructType, TimestampType

result_schema = StructType([
    StructField("antecedent", ArrayType(LongType()), False),
    StructField("consequent", ArrayType(LongType()), False),
    StructField("support", DoubleType(), False),
    StructField("confidence", DoubleType(), False),
    StructField("lift", DoubleType(), False),
    StructField("computed_at", TimestampType(), False),
])

frequent_itemsets_count = 0
total_rules = 0
result_df = spark.createDataFrame([], result_schema)

if model is not None:
    frequent_itemsets = (
        model.freqItemsets
        .withColumn("items", canonicalize_itemset(F.col("items")))
        .select("items", "freq")
    )
    frequent_itemsets_count = frequent_itemsets.count()
    print(f"\nFrequent itemsets found: {frequent_itemsets_count:,}")

    association_rules = (
        model.associationRules
        .select(
            canonicalize_itemset(F.col("antecedent")).alias("antecedent"),
            canonicalize_itemset(F.col("consequent")).alias("consequent"),
            F.col("confidence").cast("double").alias("confidence"),
        )
        .filter(
            (F.size(F.col("antecedent")) == 1)
            & (F.size(F.col("consequent")) == 1)
        )
        .withColumn(
            "rule_items",
            canonicalize_itemset(F.array_union(F.col("antecedent"), F.col("consequent"))),
        )
    )
    total_rules = association_rules.count()
    print(f"Association rules found: {total_rules:,}")

    if total_rules > 0:
        rule_itemset_frequency = frequent_itemsets.select(
            F.col("items").alias("rule_items_key"),
            F.col("freq").alias("rule_itemset_freq"),
        )
        consequent_frequency = frequent_itemsets.select(
            F.col("items").alias("consequent_key"),
            F.col("freq").alias("consequent_freq"),
        )

        rules_with_metrics = (
            association_rules
            .join(
                rule_itemset_frequency,
                F.col("rule_items") == F.col("rule_items_key"),
                "inner",
            )
            .join(
                consequent_frequency,
                F.col("consequent") == F.col("consequent_key"),
                "left",
            )
            .withColumn(
                "support",
                F.col("rule_itemset_freq") / F.lit(float(total_baskets)),
            )
            .withColumn(
                "consequent_support",
                F.col("consequent_freq") / F.lit(float(total_baskets)),
            )
            .withColumn(
                "lift",
                F.when(
                    F.col("consequent_support") > 0,
                    F.col("confidence") / F.col("consequent_support"),
                ).otherwise(F.lit(0.0)),
            )
            .select("antecedent", "consequent", "support", "confidence", "lift")
        )

        result_df = (
            rules_with_metrics
            .orderBy(F.desc("lift"), F.desc("support"), "antecedent", "consequent")
            .limit(TOP_N_RULES)
            .withColumn("computed_at", F.lit(datetime.now(timezone.utc)))
        )
        print(f"\nTop {TOP_N_RULES} association rules by lift prepared")
    else:
        print("\nNo association rules met the configured thresholds; publishing empty outputs.")
else:
    print("\nNo frequent itemsets or association rules; publishing empty outputs.")

In [ ]:
print("\n" + "="*60)
print("SAVING TO GOLD LAYER")
print("="*60)

# Save association rules to the configured gold table
result_df = validate_ml_output(result_df, "product_associations")
rules_table, rules_saved_count = save_gold(result_df, PRODUCT_ASSOCIATIONS_TABLE)
saved_rules_df = spark.table(rules_table)

print("\nSample association rules (top 10 by lift):")
if saved_rules_df.limit(1).count() > 0:
    saved_rules_df.orderBy(F.desc("lift")).limit(10).show(truncate=False)
else:
    print("  No rules to display")

In [ ]:
print("\n" + "="*60)
print("CREATING PRODUCT RECOMMENDATION TABLE")
print("="*60)

rules_df = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME)
recommendations = (
    rules_df
    .withColumn("antecedent_product", F.explode("antecedent"))
    .withColumn("consequent_product", F.explode("consequent"))
    .select(
        F.col("antecedent_product").alias("product_id"),
        F.col("consequent_product").alias("recommended_product_id"),
        "support",
        "confidence",
        "lift",
        "computed_at",
    )
)

# Singleton rules map one-to-one to recommendation pairs, so every metric
# remains from the same mined rule.
recommendations = recommendations.orderBy(
    "product_id", "recommended_product_id"
)
recommendations = validate_ml_output(recommendations, "product_recommendations")

recommendations_table, recommendations_count = save_gold(
    recommendations,
    PRODUCT_RECOMMENDATIONS_TABLE,
)

if recommendations_count > 0:
    print("\nSample recommendations (top 10):")
    (
        spark.table(recommendations_table)
        .orderBy("product_id", F.desc("lift"))
        .limit(10)
        .show(truncate=False)
    )
else:
    print("  No recommendations to display; empty table replaced any prior results.")

In [ ]:
# Log results to MLflow
with mlflow.start_run(
    run_name=f"fpgrowth_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "fpgrowth",
        "min_support": MIN_SUPPORT,
        "min_confidence": MIN_CONFIDENCE,
        "top_n_rules": TOP_N_RULES,
    })

    rules_count = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME).count()
    mlflow.log_metrics({
        "rules_saved": rules_count,
    })

    print(f"MLflow run: {run.info.run_id}")
    print(f"Rules logged: {rules_count}")


In [ ]:
print("\n" + "="*60)
print("MARKET BASKET ANALYSIS COMPLETE")
print("="*60)

# Summary statistics
rules_saved_count = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME).count()
recommendations_count = (
    spark.table(PRODUCT_RECOMMENDATIONS_TABLE_NAME).count()
    if spark.catalog.tableExists(PRODUCT_RECOMMENDATIONS_TABLE_NAME)
    else 0
)
print(f"\nSummary:")
print(f"  Transaction baskets analyzed: {total_baskets:,}")
print(f"  Frequent itemsets: {frequent_itemsets_count:,}")
print(f"  Association rules (all): {total_rules:,}")
print(f"  Top rules saved: {rules_saved_count}")
print(f"  Product recommendations saved: {recommendations_count}")

# Show Gold tables
gold_tables = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{GOLD_DB}").collect()
print(f"\nGold ({GOLD_DB}): {len(gold_tables)} tables")